In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "data").exists():
    root = root.parent

data_path = root / "data" / "reviews.csv"

print("Dataset path:", data_path.resolve())
print("File size:", data_path.stat().st_size, "bytes")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

duplicate_count_before = int(df.duplicated(subset=["category", "rating", "label", "text_"]).sum())
print("Duplicate rows before removal:", duplicate_count_before)

if duplicate_count_before > 0:
    df = df.drop_duplicates(ignore_index=True)
    print("Duplicate rows removed. Remaining rows:", len(df))
else:
    print("No duplicate rows found.")

display(df.head())


Dataset path: C:\Users\poona\OneDrive\Desktop\parkh\data\reviews.csv
File size: 15324616 bytes
Dataset shape: (40432, 4)
Columns: ['category', 'rating', 'label', 'text_']
Duplicate rows before removal: 12
Duplicate rows removed. Remaining rows: 40420


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [2]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows before removal:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))
print("Duplicate rows remaining:", df.duplicated().sum())

Duplicate rows before removal: 0
Rows after removing duplicates: 40420
Duplicate rows remaining: 0


In [3]:
%pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.model_selection import train_test_split

# Use the deduplicated dataframe, then split consistently.
X = df["text_"].astype(str)
y = df["label"].astype(str).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))
print("\nTraining labels:")
print(y_train.value_counts())


Training reviews: 32336
Testing reviews: 8084

Training labels:
label
OR    16172
CG    16164
Name: count, dtype: int64


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model_A = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=50000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_A.fit(X_train, y_train)

y_pred_A = model_A.predict(X_test)

print("Baseline Model A: Raw Review Text")
print("Accuracy:", round(accuracy_score(y_test, y_pred_A), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_A))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_A, labels=["CG", "OR"]))

Baseline Model A: Raw Review Text
Accuracy: 0.9332

Classification Report:
              precision    recall  f1-score   support

          CG       0.95      0.92      0.93      4041
          OR       0.92      0.95      0.93      4043

    accuracy                           0.93      8084
   macro avg       0.93      0.93      0.93      8084
weighted avg       0.93      0.93      0.93      8084


Confusion Matrix:
[[3717  324]
 [ 216 3827]]


In [6]:
from sklearn.model_selection import train_test_split

X = df["text_"].astype(str)
y = df["label"].astype(str).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 32336
Testing samples: 8084


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model_A = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=50000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_A.fit(X_train, y_train)

y_pred_A = model_A.predict(X_test)

print("Model A: Raw Review Text")
print("Accuracy:", round(accuracy_score(y_test, y_pred_A), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_A))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_A, labels=["CG", "OR"]))

Model A: Raw Review Text
Accuracy: 0.9332

Classification Report:
              precision    recall  f1-score   support

          CG       0.95      0.92      0.93      4041
          OR       0.92      0.95      0.93      4043

    accuracy                           0.93      8084
   macro avg       0.93      0.93      0.93      8084
weighted avg       0.93      0.93      0.93      8084


Confusion Matrix:
[[3717  324]
 [ 216 3827]]


In [8]:
import re

def fix_artifacts(text):
    text = str(text).replace("\u00a0", " ")
    text = text.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    text = re.sub(r"([.!?])(?=[A-Za-z])", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

X_train_clean = X_train.apply(fix_artifacts)
X_test_clean = X_test.apply(fix_artifacts)

changed_count = int(
    (X_train.reset_index(drop=True) != X_train_clean.reset_index(drop=True)).sum()
)

print("Training reviews:", len(X_train))
print("Reviews changed:", changed_count)
print("Percentage changed:", f"{changed_count / len(X_train) * 100:.2f}%")


Training reviews: 32336
Reviews changed: 18564
Percentage changed: 57.41%


In [9]:
changed_count = (
    X_train.reset_index(drop=True)
    != X_train_clean.reset_index(drop=True)
).sum()

print("Training reviews:", len(X_train))
print("Reviews changed:", changed_count)
print(
    "Percentage changed:",
    f"{changed_count / len(X_train) * 100:.2f}%"
)

Training reviews: 32336
Reviews changed: 18564
Percentage changed: 57.41%


In [10]:
print("Rows in df:", len(df))
print("Duplicate rows after removal:", df.duplicated(subset=["category", "rating", "label", "text_"]).sum())


Rows in df: 40420
Duplicate rows after removal: 0


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model_B = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=50000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_B.fit(X_train_clean, y_train)

y_pred_B = model_B.predict(X_test_clean)

print("Model B: Artifact-Neutralized Text")
print("Accuracy:", round(accuracy_score(y_test, y_pred_B), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_B))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_B, labels=["CG", "OR"]))

Model B: Artifact-Neutralized Text
Accuracy: 0.9332

Classification Report:
              precision    recall  f1-score   support

          CG       0.95      0.92      0.93      4041
          OR       0.92      0.95      0.93      4043

    accuracy                           0.93      8084
   macro avg       0.93      0.93      0.93      8084
weighted avg       0.93      0.93      0.93      8084


Confusion Matrix:
[[3717  324]
 [ 216 3827]]


In [24]:

# Inspect examples changed by the cleaning function
changed_mask = (
    X_train.reset_index(drop=True)
    != X_train_clean.reset_index(drop=True)
)

changed_indices = changed_mask[changed_mask].index[:5]

print("Total changed reviews:", int(changed_mask.sum()))
print("\n--- Examples of cleaning changes ---")

for i in changed_indices:
    print(f"\nExample {i + 1}")
    print("BEFORE:", repr(X_train.reset_index(drop=True).iloc[i]))
    print("AFTER: ", repr(X_train_clean.reset_index(drop=True).iloc[i]))

# Verify whether the cleaning changes any test review
test_changed = (
    X_test.reset_index(drop=True)
    != X_test_clean.reset_index(drop=True)
).sum()

print("\nTest reviews changed:", int(test_changed))


Total changed reviews: 18564

--- Examples of cleaning changes ---

Example 1
BEFORE: 'Much like the card case this is a well built wallet. The seams seem strong as well as the leather itself. I love the variety of colors this company offers as well. This would be a great option if you were traveling and needed something easy to hide or just take up as little space as possible. This is a very slim style card wallet and still as the internal slot for some cash as well. I like having both options this company offers for that reason. OPTIONS! I can down size if need be like if were were going to an amusement park or zoo, etc I can use this wallet but for everyday shopping the card case is my go to!\n\nI received this at a discount or at no cost for my review and feedback, however my opinion is 100% my own, honest and uninfluenced in any way.'
AFTER:  'Much like the card case this is a well built wallet. The seams seem strong as well as the leather itself. I love the variety of colors this

In [12]:
accuracy_A = accuracy_score(y_test, y_pred_A)
accuracy_B = accuracy_score(y_test, y_pred_B)

print(f"Model A — Raw text:     {accuracy_A:.4f}")
print(f"Model B — Cleaned text: {accuracy_B:.4f}")
print(f"Accuracy difference:   {accuracy_B - accuracy_A:+.4f}")

Model A — Raw text:     0.9332
Model B — Cleaned text: 0.9332
Accuracy difference:   +0.0000


In [13]:
import re
import pandas as pd

def extract_features(text):
    text = str(text)

    return {
        "word_count": len(text.split()),
        "ends_with_punctuation": int(
            bool(re.search(r"[.!?]\s*$", text))
        ),
        "missing_space_after_punctuation": int(
            bool(re.search(r"[.!?][A-Za-z]", text))
        )
    }

features = pd.DataFrame(
    [extract_features(text) for text in df["text_"]],
    index=df.index
)

display(features.head())
print("Feature shape:", features.shape)

,word_count,ends_with_punctuation,missing_space_after_punctuation
0,12,0,1
1,16,0,0
2,14,1,0
3,17,0,0
4,18,0,0


Feature shape: (40420, 3)


In [14]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

X_train = train_df["text_"].astype(str)
X_test = test_df["text_"].astype(str)
y_train = train_df["label"].astype(str).str.strip()
y_test = test_df["label"].astype(str).str.strip()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


Training rows: 32336
Testing rows: 8084


In [15]:
X_train_features = features.loc[train_df.index]
X_test_features = features.loc[test_df.index]

print("Training features:", X_train_features.shape)
print("Testing features:", X_test_features.shape)

Training features: (32336, 3)
Testing features: (8084, 3)


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model_C = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_C.fit(X_train_features, y_train)

y_pred_C = model_C.predict(X_test_features)

print("Model C: Formatting Features Only")
print("Accuracy:", round(accuracy_score(y_test, y_pred_C), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_C))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_C, labels=["CG", "OR"]))

Model C: Formatting Features Only
Accuracy: 0.7768

Classification Report:
              precision    recall  f1-score   support

          CG       0.83      0.70      0.76      4041
          OR       0.74      0.86      0.79      4043

    accuracy                           0.78      8084
   macro avg       0.78      0.78      0.78      8084
weighted avg       0.78      0.78      0.78      8084


Confusion Matrix:
[[2813 1228]
 [ 576 3467]]


In [17]:
comparison = pd.DataFrame({
    "Model": [
        "A - Raw Text",
        "B - Cleaned Text",
        "C - Formatting Only"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_A),
        accuracy_score(y_test, y_pred_B),
        accuracy_score(y_test, y_pred_C)
    ]
})

display(comparison.round(4))

,Model,Accuracy
0,A - Raw Text,0.9332
1,B - Cleaned Text,0.9332
2,C - Formatting Only,0.7768


In [18]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "model_comparison.csv",
    index=False
)

print("Results saved to:", (output_dir / "model_comparison.csv").resolve())
display(comparison.round(4))

Results saved to: C:\Users\poona\OneDrive\Desktop\parkh\outputs\model_comparison.csv


,Model,Accuracy
0,A - Raw Text,0.9332
1,B - Cleaned Text,0.9332
2,C - Formatting Only,0.7768


In [19]:
from pathlib import Path

if "comparison" not in globals():
    raise NameError("Run the model evaluation cells before saving comparison.csv.")

output_dir = Path.cwd()
if output_dir.name == "notebooks":
    output_dir = output_dir.parent

output_dir = output_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "model_comparison.csv"
comparison.to_csv(output_file, index=False)

print("Saved comparison results to:", output_file.resolve())


Saved comparison results to: C:\Users\poona\OneDrive\Desktop\parkh\outputs\model_comparison.csv


In [20]:
import numpy as np

different_predictions = np.sum(y_pred_A != y_pred_B)

print("Test reviews:", len(y_test))
print("Different predictions:", different_predictions)
print(
    "Prediction disagreement:",
    f"{different_predictions / len(y_test) * 100:.2f}%"
)

Test reviews: 8084
Different predictions: 0
Prediction disagreement: 0.00%


In [21]:
import pandas as pd

errors = pd.DataFrame({
    "review": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(y_pred_A).reset_index(drop=True)
})

errors = errors[errors["actual"] != errors["predicted"]]

print("Total incorrect predictions:", len(errors))
display(errors.head(10))

Total incorrect predictions: 540


,review,actual,predicted
16,A good value at 20 bucks. The sound affects m...,OR,CG
27,Would recommend----looking forward to his next...,CG,OR
31,Good Memory card! But prices are a little lowe...,CG,OR
69,Xander has control in all areas of his life an...,CG,OR
77,Ears were really bad and had an odor. I was c...,CG,OR
88,"I've used this book for years, and I swear by ...",OR,CG
145,Her eyes lit up when she opened the box and sa...,CG,OR
213,Every single one of these books has me on the ...,CG,OR
225,I love the size and look of these things I ha...,OR,CG
228,This is a cassic and it is so funny. I find t...,OR,CG


In [22]:
print("Total incorrect predictions:", len(errors))

print("\nActual label vs predicted label:")
display(
    errors.groupby(["actual", "predicted"])
    .size()
    .reset_index(name="count")
)

Total incorrect predictions: 540

Actual label vs predicted label:


,actual,predicted,count
0,CG,OR,324
1,OR,CG,216


In [23]:
print(
    "Rows where A and B predictions differ:",
    (y_pred_A != y_pred_B).sum()
)

vec = model_B.named_steps["tfidf"]

diff = (
    vec.transform(X_test_clean)
    != vec.transform(X_test)
).nnz

print(
    "TF-IDF feature cells that differ:",
    diff
)

Rows where A and B predictions differ: 0
TF-IDF feature cells that differ: 0
